## Train a topic model with Perch2 embeddings and PCEN features
---
This notebook concatenates Perch2 embeddings with PCEN spectrogram features, then trains a topic model to uncover recurring sound phrases. It is intended for humpback song from *Pacific Ocean Sound* recordings, using both a learned acoustic embedding and a classical PCEN representation.

Author: Danelle Cline dcline@mbari.org

### Load configuration
Create the output directory, then load and verify `config.yaml`. The config supplies window sizes, topic-model hyperparameters, and paths used in later cells.

In [ ]:
from pathlib import Path

output_path = Path("output_hb")
dataset_path = Path("dataset_hb")
output_path.mkdir(parents=True, exist_ok=True)

from stm.config import Config
try:
    config = Config.load(
        Path("../config.yaml"),
        wav_path=dataset_path,
        output_path=output_path,
    )
    config.verify()
except Exception as e:
    print(e)

### Extract PCEN and Perch2 features
Measure the dataset duration, build a regular Perch2 time grid, extract PCEN features, and embed audio windows with the Perch2 ONNX model. Both feature blocks are used together in the next cell.

In [ ]:
from pathlib import Path
from stm.topicmodel import TopicModelRunner
from stm.features import PcenExtractor, TimeGrid, Perch2Extractor
from stm.embed import total_audio_seconds

duration = total_audio_seconds(dataset_path)
n = total_audio_seconds(dataset_path)
n_windows = int((duration - config.perch_audio_seconds) // config.perch_hop_seconds) + 1

print(f"Processing {duration} total audio seconds")

perch_grid = TimeGrid.regular(
    start=0.0,
    hop=config.perch_hop_seconds,
    window=config.perch_audio_seconds,
    n=n_windows,
)

pcen_block = PcenExtractor.from_config(config).extract(dataset_path)

perch_extractor = Perch2Extractor.from_config(
    config,
    model=Path("./perch_v2.onnx"),
)
perch_block = perch_extractor.extract(source=dataset_path, grid=perch_grid)

### Train the topic model
Fit a topic model on the concatenated PCEN and Perch2 blocks. Features are aligned to the PCEN time grid so both representations occupy the same documents.

In [ ]:
runner = TopicModelRunner(
    timeout=6000,
    alpha=config.alpha,
    beta=config.beta,
    gamma=config.gamma,
    num_topics=None,
    document_seconds=0.32,
)
result = runner.run_from_block(
    [pcen_block, perch_block],
    config.doc_path,
    config.model_path,
    target=pcen_block.grid,
)

### Report model diagnostics
Print average perplexity and the path to the maximum-likelihood topic-over-time file.

In [ ]:
print(result.avg_perplexity, result.maxlikelihood_with_time_path)

### Plot topics on spectrograms
Overlay inferred topics on spectrogram chunks of the source audio for visual inspection.

In [ ]:
from stm.topicmodel.plotter import Plotter

plotter = Plotter(model_dir=config.model_path, config=config)
plotter.plot(dataset_path, n_chunks=10, chunk_size=60, freq_range=(0, 4000), window_size=1024)